# ViT-Tiny — Q8.8 Fixed-Point Implementation

**Format: Q8.8 — 16-bit fixed point**
- Total bits: 16 (stored as `int32` in Python to allow overflow headroom during MAC)
- Integer bits: 8 (range −128 to +127)
- Fractional bits: 8
- Scale: `SCALE = 2^8 = 256`
- Resolution: `1/256 ≈ 0.0039`
- Real value: `fixed_int / 256`

This matches `W1.mem` / `W2.mem` already in the project.

**Accumulation:** MAC results are kept in `int32` during dot products, then right-shifted by 8 bits (`>> 8`) to return to Q8.8.

**Comparison:** The `timm` float32 model loaded from `vit_malware.pth` is the golden reference.

## 0 — Imports & constants

In [ ]:
import math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# ── Q8.8 parameters ──────────────────────────────────────────────────────────
FRAC_BITS = 8
SCALE     = 1 << FRAC_BITS   # 256
Q_MAX     = (1 << 15) - 1    # +32767
Q_MIN     = -(1 << 15)       # -32768

# ── ViT-Tiny architecture constants ──────────────────────────────────────────
IMG_SIZE    = 224
PATCH_SIZE  = 16
EMBED_DIM   = 192
DEPTH       = 12
NUM_HEADS   = 3
HEAD_DIM    = EMBED_DIM // NUM_HEADS   # 64
MLP_HIDDEN  = EMBED_DIM * 4            # 768
NUM_CLASSES = 2
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 196
NUM_TOKENS  = NUM_PATCHES + 1                 # 197
ATTN_SCALE  = 1.0 / math.sqrt(HEAD_DIM)      # 0.125

print(f"Q8.8  scale : {SCALE}")
print(f"Q8.8  range : [{Q_MIN}, {Q_MAX}]")
print(f"Attn scale  : {ATTN_SCALE}  (Q8.8 int = {round(ATTN_SCALE*SCALE)})")

## 1 — Q8.8 helper functions

In [ ]:
def to_fixed(x: torch.Tensor) -> torch.Tensor:
    """float32 → Q8.8 int32 (clipped to int16 range)"""
    return (x.float() * SCALE).round().clamp(Q_MIN, Q_MAX).to(torch.int32)

def to_float(x: torch.Tensor) -> torch.Tensor:
    """Q8.8 int32 → float32"""
    return x.to(torch.float32) / SCALE

def q_clip(x: torch.Tensor) -> torch.Tensor:
    """Clip int32 accumulator back to int16 range"""
    return x.clamp(Q_MIN, Q_MAX)

def q_mac(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Fixed-point 2-D matrix multiply.
    Both inputs are Q8.8 int32.  Result is Q8.8 int32.
    Internally: float(a) @ float(b) / SCALE  (= integer >>8 after accumulate).
    Both a and b must be exactly 2-D.
    """
    assert a.dim() == 2 and b.dim() == 2, f"q_mac expects 2-D tensors, got {a.shape} and {b.shape}"
    result = (a.float() @ b.float()) / SCALE
    return result.round().to(torch.int32)

# ── Sanity test ──
_a = to_fixed(torch.tensor([[1.5, -0.5]]))
_b = to_fixed(torch.tensor([[2.0], [0.5]]))
_c = q_mac(_a, _b)
print(f"MAC test: 1.5*2.0 + (-0.5)*0.5 = {1.5*2.0 + (-0.5)*0.5}")
print(f"  Fixed result : {_c.item()}  → {to_float(_c).item():.4f}  (expect 2.75)")

## 2 — Load trained weights → Q8.8

In [ ]:
WEIGHTS_PATH = "vit_malware.pth"
assert os.path.exists(WEIGHTS_PATH), f"'{WEIGHTS_PATH}' not found"

ref_model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
ref_model.head = nn.Linear(ref_model.head.in_features, NUM_CLASSES)
ref_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu'))
ref_model.eval()
sd = ref_model.state_dict()

def wq(key):  # weight as Q8.8 int32
    return to_fixed(sd[key])
def wf(key):  # weight as float32
    return sd[key].clone()

print(f"Loaded '{WEIGHTS_PATH}'")

# Cross-check against existing .mem files
for fname, key in [("W1.mem", "blocks.0.mlp.fc1.weight"),
                   ("W2.mem", "blocks.0.mlp.fc2.weight")]:
    if os.path.exists(fname):
        mem = np.loadtxt(fname, dtype=int)
        match = np.array_equal(mem, wq(key).numpy().flatten())
        print(f"{fname} cross-check: {'✅ PASS' if match else '❌ MISMATCH'}")

## 3 — Fixed-Point Patch Embedding

In [ ]:
class FxPatchEmbed:
    """Conv2d(3,192,16,stride=16) → flatten → (1,196,192) in Q8.8"""
    def __init__(self):
        self.W = to_fixed(wf('patch_embed.proj.weight').reshape(EMBED_DIM, -1))  # (192,768)
        self.b = to_fixed(wf('patch_embed.proj.bias'))                           # (192,)

    def forward(self, x_q):   # (1,3,224,224) Q8.8
        patches = F.unfold(to_float(x_q), kernel_size=PATCH_SIZE, stride=PATCH_SIZE)  # (1,768,196)
        patches_q = to_fixed(patches[0].T)                   # (196,768) Q8.8
        out = q_mac(patches_q, self.W.T)                     # (196,192) Q8.8
        out = q_clip(out + self.b.unsqueeze(0))              # add bias
        return out.unsqueeze(0)                              # (1,196,192)


_pe  = FxPatchEmbed()
_img = to_fixed(torch.randn(1, 3, 224, 224))
_out = _pe.forward(_img)
print(f"FxPatchEmbed → {list(_out.shape)}  (expect (1,196,192))")
print(f"Float range  : [{to_float(_out).min():.3f}, {to_float(_out).max():.3f}]")

## 4 — Fixed-Point Layer Norm

In [ ]:
class FxLayerNorm:
    """
    Q8.8 Layer Norm over last dim (192).
    mean/var computed in float (RTL: fixed-point divider + inv-sqrt LUT).
    """
    def __init__(self, gamma_key, beta_key, eps=1e-6):
        self.gamma_f = wf(gamma_key)  # keep float for element-wise multiply
        self.beta_q  = to_fixed(wf(beta_key))   # (192,) Q8.8
        self.eps     = eps

    def forward(self, x_q):   # (...,192) Q8.8
        x_f    = to_float(x_q)
        mean   = x_f.mean(dim=-1, keepdim=True)
        var    = x_f.var(dim=-1, unbiased=False, keepdim=True)
        x_norm = (x_f - mean) / torch.sqrt(var + self.eps)  # float normalised
        # Apply gamma (float multiply then convert) + beta (Q8.8 add)
        out_f  = x_norm * self.gamma_f                      # element-wise
        out_q  = to_fixed(out_f)                            # → Q8.8
        return q_clip(out_q + self.beta_q)                  # add beta


_ln  = FxLayerNorm('blocks.0.norm1.weight', 'blocks.0.norm1.bias')
_x   = to_fixed(torch.randn(1, 197, 192))
_out = _ln.forward(_x)
print(f"FxLayerNorm → {list(_out.shape)}")
print(f"Per-token mean≈0 : {to_float(_out[0,0]).mean().item():.4f}")
print(f"Per-token std ≈1 : {to_float(_out[0,0]).std().item():.4f}")

## 5 — Fixed-Point GELU (LUT)

In [ ]:
LUT_SIZE   = 512
LUT_IN_MIN = -4.0
LUT_IN_MAX = +4.0
LUT_STEP   = (LUT_IN_MAX - LUT_IN_MIN) / LUT_SIZE
GELU_LUT   = to_fixed(F.gelu(torch.linspace(LUT_IN_MIN, LUT_IN_MAX, LUT_SIZE)))

def fx_gelu(x_q: torch.Tensor) -> torch.Tensor:
    """Q8.8 GELU via 512-entry LUT covering [-4, +4]"""
    idx = ((to_float(x_q) - LUT_IN_MIN) / LUT_STEP).long().clamp(0, LUT_SIZE - 1)
    return GELU_LUT[idx]

print("GELU LUT accuracy:")
for v in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    q_in  = to_fixed(torch.tensor([v]))
    approx = to_float(fx_gelu(q_in)).item()
    ref    = F.gelu(torch.tensor(v)).item()
    print(f"  GELU({v:+.1f})  ref={ref:+.4f}  Q8.8={approx:+.4f}  err={abs(ref-approx):.4f}")

## 6 — Fixed-Point MLP Block

In [ ]:
class FxMLP:
    """Q8.8  fc1(192→768) → GELU → fc2(768→192)"""
    def __init__(self, idx):
        p = f'blocks.{idx}.mlp'
        self.W1 = to_fixed(wf(f'{p}.fc1.weight'))  # (768,192)
        self.b1 = to_fixed(wf(f'{p}.fc1.bias'))    # (768,)
        self.W2 = to_fixed(wf(f'{p}.fc2.weight'))  # (192,768)
        self.b2 = to_fixed(wf(f'{p}.fc2.bias'))    # (192,)

    def forward(self, x_q):   # (B,N,192) Q8.8
        B, N, _ = x_q.shape
        x = x_q.reshape(B * N, EMBED_DIM)          # (B*N,192)
        h  = q_clip(q_mac(x,  self.W1.T) + self.b1)  # (B*N,768)
        h  = fx_gelu(h)                               # (B*N,768)
        out = q_clip(q_mac(h, self.W2.T) + self.b2)  # (B*N,192)
        return out.reshape(B, N, EMBED_DIM)

_mlp = FxMLP(0)
_x   = to_fixed(torch.randn(1, 197, 192))
_out = _mlp.forward(_x)
print(f"FxMLP → {list(_out.shape)}  (expect (1,197,192))")

## 7 — Fixed-Point Softmax (exp LUT)

In [ ]:
EXP_LUT_SIZE = 512
EXP_IN_MIN   = -8.0
EXP_IN_MAX   =  0.0
EXP_LUT_STEP = (EXP_IN_MAX - EXP_IN_MIN) / EXP_LUT_SIZE
EXP_LUT      = to_fixed(torch.exp(torch.linspace(EXP_IN_MIN, EXP_IN_MAX, EXP_LUT_SIZE)))

def fx_exp(x_q):
    idx = ((to_float(x_q) - EXP_IN_MIN) / EXP_LUT_STEP).long().clamp(0, EXP_LUT_SIZE - 1)
    return EXP_LUT[idx]

def fx_softmax(scores_q):   # (..., N) int32
    """Subtract-max → exp LUT → normalise"""
    shifted  = q_clip(scores_q - scores_q.max(dim=-1, keepdim=True).values)
    exp_vals = fx_exp(shifted).to(torch.int32)
    exp_f    = to_float(exp_vals)
    sm_f     = exp_f / (exp_f.sum(dim=-1, keepdim=True) + 1e-9)
    return to_fixed(sm_f)

_s  = to_fixed(torch.tensor([[1.0, 2.0, 0.5]]))
_sm = fx_softmax(_s)
ref = F.softmax(torch.tensor([[1.0, 2.0, 0.5]]), dim=-1)
print(f"Softmax ref  : {ref.squeeze().tolist()}")
print(f"Softmax Q8.8 : {to_float(_sm).squeeze().tolist()}")
print(f"Max error    : {(ref - to_float(_sm)).abs().max().item():.4f}")

## 8 — Fixed-Point Multi-Head Self-Attention  *(fixed)*

Key fix from original: scores are computed **per head** using a loop so the Q@Kᵀ
matmul is always `(N, head_dim) @ (head_dim, N) → (N, N)`, never collapsing heads.

In [ ]:
class FxMHSA:
    """
    Q8.8 Multi-Head Self-Attention.
    3 heads, head_dim=64, embed_dim=192.

    All heavy matmuls use q_mac (2-D only).
    Head loop keeps dimensions explicit and RTL-friendly.
    """
    def __init__(self, idx):
        p = f'blocks.{idx}.attn'
        self.qkv_w  = to_fixed(wf(f'{p}.qkv.weight'))   # (576,192)
        self.qkv_b  = to_fixed(wf(f'{p}.qkv.bias'))     # (576,)
        self.proj_w = to_fixed(wf(f'{p}.proj.weight'))   # (192,192)
        self.proj_b = to_fixed(wf(f'{p}.proj.bias'))     # (192,)

    def forward(self, x_q):   # (B, N, 192) Q8.8
        B, N, C = x_q.shape   # typically B=1, N=197, C=192

        # ── Step 1: Fused QKV projection ──────────────────────────────────
        # (B*N, 192) @ (192, 576) = (B*N, 576)  Q8.8
        x_2d = x_q.reshape(B * N, C)
        qkv  = q_clip(q_mac(x_2d, self.qkv_w.T) + self.qkv_b)   # (B*N, 576)

        # Reshape: (B, N, 3, H, D) then split Q/K/V
        qkv = qkv.reshape(B, N, 3, NUM_HEADS, HEAD_DIM).permute(2, 0, 3, 1, 4)
        # Q, K, V each: (B, NUM_HEADS, N, HEAD_DIM)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ── Step 2-4: Per-head scaled dot-product attention ────────────────
        # Each head: Q_h=(B,N,D), K_h=(B,N,D), V_h=(B,N,D)
        # Scores = Q_h @ K_h.T * scale  →  (B, N, N)  per head
        ctx_heads = []
        for h in range(NUM_HEADS):
            for b in range(B):
                q_h = Q[b, h]   # (N, D)  Q8.8
                k_h = K[b, h]   # (N, D)  Q8.8
                v_h = V[b, h]   # (N, D)  Q8.8

                # (N,D) @ (D,N) = (N,N) — properly shaped 2-D matmul
                scores = q_mac(q_h, k_h.T)    # (N, N) Q8.8

                # Scale by 1/sqrt(64) = 0.125  →  in Q8.8: multiply by 32/256
                scores = (scores.float() * ATTN_SCALE).round().to(torch.int32)
                scores = q_clip(scores)

                # Softmax row-wise
                attn = fx_softmax(scores)     # (N, N) Q8.8

                # Context: (N,N) @ (N,D) = (N,D)
                ctx = q_mac(attn, v_h)        # (N, D) Q8.8
                ctx_heads.append(ctx)

        # ── Step 5: Merge heads (B, N, H*D) = (B, N, 192) ─────────────────
        # ctx_heads is list of B*NUM_HEADS tensors each (N, D)
        # Re-order from [h0b0, h1b0, h2b0] to (B, N, C)
        merged = torch.zeros(B, N, C, dtype=torch.int32)
        for b in range(B):
            for h in range(NUM_HEADS):
                merged[b, :, h*HEAD_DIM:(h+1)*HEAD_DIM] = ctx_heads[b*NUM_HEADS + h]

        # ── Step 6: Output projection ──────────────────────────────────────
        out_2d = q_clip(q_mac(merged.reshape(B*N, C), self.proj_w.T) + self.proj_b)
        return out_2d.reshape(B, N, C)


_attn = FxMHSA(0)
_x    = to_fixed(torch.randn(1, 197, 192))
_out  = _attn.forward(_x)
print(f"FxMHSA → {list(_out.shape)}  (expect (1,197,192))")
print(f"Float range : [{to_float(_out).min():.3f}, {to_float(_out).max():.3f}]")

## 9 — Fixed-Point Transformer Block

In [ ]:
class FxTransformerBlock:
    """
    Q8.8 block:
        x = x + MHSA(LN1(x))
        x = x + MLP(LN2(x))
    """
    def __init__(self, idx):
        self.norm1 = FxLayerNorm(f'blocks.{idx}.norm1.weight', f'blocks.{idx}.norm1.bias')
        self.attn  = FxMHSA(idx)
        self.norm2 = FxLayerNorm(f'blocks.{idx}.norm2.weight', f'blocks.{idx}.norm2.bias')
        self.mlp   = FxMLP(idx)

    def forward(self, x_q):
        x_q = q_clip(x_q + self.attn.forward(self.norm1.forward(x_q)))
        x_q = q_clip(x_q + self.mlp.forward(self.norm2.forward(x_q)))
        return x_q


_blk = FxTransformerBlock(0)
_x   = to_fixed(torch.randn(1, 197, 192))
_out = _blk.forward(_x)
print(f"FxTransformerBlock → {list(_out.shape)}  (expect (1,197,192))")

## 10 — Complete Q8.8 ViT Forward Pass

In [ ]:
class FxViT:
    """Complete Q8.8 ViT-Tiny malware classifier"""
    def __init__(self):
        print("Loading FxViT weights...")
        self.patch_embed = FxPatchEmbed()
        self.cls_token   = to_fixed(wf('cls_token'))   # (1,1,192)
        self.pos_embed   = to_fixed(wf('pos_embed'))   # (1,197,192)
        self.blocks      = [FxTransformerBlock(i) for i in range(DEPTH)]
        self.norm        = FxLayerNorm('norm.weight', 'norm.bias')
        self.head_w      = to_fixed(wf('head.weight'))  # (2,192)
        self.head_b      = to_fixed(wf('head.bias'))    # (2,)
        print("Done.")

    def forward(self, x_q):   # (1,3,224,224) Q8.8
        x   = self.patch_embed.forward(x_q)                       # (1,196,192)
        cls = self.cls_token.expand(x.shape[0], -1, -1)           # (1,1,192)
        x   = q_clip(torch.cat([cls, x], dim=1) + self.pos_embed) # (1,197,192)
        for blk in self.blocks:
            x = blk.forward(x)
        x   = self.norm.forward(x)          # (1,197,192)
        cls = x[:, 0]                       # (1,192)  CLS token
        logits = q_clip(q_mac(cls, self.head_w.T) + self.head_b)  # (1,2)
        return logits


fx_vit = FxViT()

## 11 — Side-by-Side Comparison: Float32 vs Q8.8

In [ ]:
import time

torch.manual_seed(42)
img_f = torch.randn(1, 3, 224, 224)
img_q = to_fixed(img_f)

# Float32 reference
ref_model.eval()
with torch.no_grad():
    t0 = time.time()
    ref_logits = ref_model(img_f)
    t_ref = time.time() - t0
ref_probs = F.softmax(ref_logits, dim=-1).squeeze()

# Q8.8 fixed-point
t0 = time.time()
fx_logits_q = fx_vit.forward(img_q)
t_fx = time.time() - t0
fx_logits_f = to_float(fx_logits_q)
fx_probs    = F.softmax(fx_logits_f, dim=-1).squeeze()

ref_class = 'malware' if ref_probs[1] > ref_probs[0] else 'benign'
fx_class  = 'malware' if fx_probs[1]  > fx_probs[0]  else 'benign'
max_diff  = (ref_logits - fx_logits_f).abs().max().item()

print("═" * 54)
print(f"  {'':20}  {'Float32':>12}  {'Q8.8':>12}")
print("─" * 54)
print(f"  {'Logit[benign]':20}  {ref_logits[0,0].item():+12.4f}  {fx_logits_f[0,0].item():+12.4f}")
print(f"  {'Logit[malware]':20}  {ref_logits[0,1].item():+12.4f}  {fx_logits_f[0,1].item():+12.4f}")
print("─" * 54)
print(f"  {'P(benign)':20}  {ref_probs[0].item():12.4f}  {fx_probs[0].item():12.4f}")
print(f"  {'P(malware)':20}  {ref_probs[1].item():12.4f}  {fx_probs[1].item():12.4f}")
print("─" * 54)
print(f"  {'Prediction':20}  {ref_class:>12}  {fx_class:>12}")
print("═" * 54)
print(f"  Max |logit diff| : {max_diff:.4f}")
print(f"  Class match      : {'✅ YES' if ref_class==fx_class else '❌ NO'}")
print(f"  Float32 time     : {t_ref*1000:.0f} ms")
print(f"  Q8.8 time (CPU)  : {t_fx*1000:.0f} ms")

## 12 — Multi-image Class Agreement Test (50 samples)

In [ ]:
torch.manual_seed(0)
N_SAMPLES   = 50
match_count = 0
max_diffs   = []

for i in range(N_SAMPLES):
    img_f = torch.randn(1, 3, 224, 224)
    img_q = to_fixed(img_f)

    with torch.no_grad():
        ref_l  = ref_model(img_f)
        fx_l_q = fx_vit.forward(img_q)
        fx_l_f = to_float(fx_l_q)

    if ref_l.argmax().item() == fx_l_f.argmax().item():
        match_count += 1
    max_diffs.append((ref_l - fx_l_f).abs().max().item())

    if (i+1) % 10 == 0:
        print(f"  [{i+1:3d}/{N_SAMPLES}]  running agreement: {match_count/(i+1)*100:.1f}%")

print()
print(f"Class agreement   : {match_count}/{N_SAMPLES} = {match_count/N_SAMPLES*100:.1f}%")
print(f"Avg |logit diff|  : {sum(max_diffs)/len(max_diffs):.4f}")
print(f"Max |logit diff|  : {max(max_diffs):.4f}")

pct = match_count / N_SAMPLES * 100
if pct >= 95:
    print("\n✅  Q8.8 is RTL-ready (≥95% class agreement)")
elif pct >= 85:
    print("\n⚠️  Marginal — consider widening activations to Q10.6")
else:
    print("\n❌  Too lossy — increase to Q12.4 or Q16")

## 13 — Per-Stage Quantisation Drift

In [ ]:
torch.manual_seed(0)
img_f = torch.randn(1, 3, 224, 224)
img_q = to_fixed(img_f)

def cmp(name, f_t, q_t):
    err = (f_t - to_float(q_t)).abs().max().item()
    rng = f"[{f_t.min():.3f}, {f_t.max():.3f}]"
    print(f"  {name:<32} range={rng:<22}  max|err|={err:.4f}")

print(f"{'Stage':<34} {'Float32 range':<22}  {'Max|err|'}")
print("-" * 75)

with torch.no_grad():
    # Patch embed
    f_x = ref_model.patch_embed(img_f)
    q_x = fx_vit.patch_embed.forward(img_q)
    cmp("PatchEmbed", f_x, q_x)

    # CLS + pos
    f_x = torch.cat([ref_model.cls_token.expand(1,-1,-1), f_x], 1) + ref_model.pos_embed
    q_x = q_clip(torch.cat([fx_vit.cls_token.expand(1,-1,-1), q_x], 1) + fx_vit.pos_embed)
    cmp("CLS + PosEmbed", f_x, q_x)

    # 12 blocks
    block_errs = []
    for i in range(DEPTH):
        f_x = ref_model.blocks[i](f_x)
        q_x = fx_vit.blocks[i].forward(q_x)
        err = (f_x - to_float(q_x)).abs().max().item()
        block_errs.append(err)
        cmp(f"Block {i:02d}", f_x, q_x)

    # Final norm
    f_n = ref_model.norm(f_x)
    q_n = fx_vit.norm.forward(q_x)
    cmp("Final LayerNorm", f_n, q_n)

    # Logits
    f_logits = ref_model.head(f_n[:, 0])
    q_logits = q_clip(q_mac(q_n[:, 0], fx_vit.head_w.T) + fx_vit.head_b)
    cmp("Logits", f_logits, q_logits)

## 14 — Error accumulation plot

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("architecture_graphs", exist_ok=True)

plt.figure(figsize=(10, 4))
plt.plot(range(DEPTH), block_errs, 'o-', color='crimson', linewidth=2, markersize=6)
plt.axhline(0.5, ls='--', color='gray', label='0.5 threshold')
plt.axhline(1.0, ls=':', color='orange', label='1.0 threshold')
plt.xlabel("Transformer Block", fontsize=12)
plt.ylabel("Max |float − Q8.8|", fontsize=12)
plt.title("Quantisation Error Accumulation per Block (Q8.8)", fontsize=13)
plt.xticks(range(DEPTH))
plt.legend()
plt.tight_layout()
plt.savefig("architecture_graphs/quant_error_per_block.png", dpi=150)
plt.show()
print("Saved → architecture_graphs/quant_error_per_block.png")

## 15 — Q8.8 Format Summary

| Element | Bits | Format | Notes |
|---|---|---|---|
| Weights (W1, W2, QKV…) | 16 | Q8.8 int16 | stored in `.mem` files |
| Activations | 16 | Q8.8 int16 | clipped after every add/mul |
| MAC accumulator | 32 | int32 Q8.8 | `>>8` after dot product |
| GELU LUT | 512 entries | Q8.8 output | 9-bit address |
| Softmax exp LUT | 512 entries | Q8.8 output | input range [−8, 0] |
| Attn scale (0.125) | — | multiply ×0.125 | exact in binary (>>3) |